In [1]:
"""
Descarga de generación eléctrica detallada por tipo de tecnología
Sistema peninsular, 01/10/2020 - 31/12/2020

Fuente: API de ESIOS (Red Eléctrica de España)
Requiere un token personal gratuito: solicítalo a consultasios@ree.es
Documentación: https://api.esios.ree.es/

El script busca automáticamente el ID de cada indicador por nombre,
descarga sus valores horarios y construye un dataset ancho con las
columnas solicitadas.
"""

import requests
import pandas as pd
import time

# ---------------------------------------------------------------
# 1) CONFIGURACIÓN
# ---------------------------------------------------------------
TOKEN = "294da85af17aff04fda25ec930700cf59bde0ac3d8322d075617fc038c1ff224"   # <-- pega aquí tu token personal de ESIOS

START_DATE = "2020-10-01T00:00:00"
END_DATE   = "2020-12-31T23:59:59"

HEADERS = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json",
    "x-api-key": TOKEN,
}

BASE_URL = "https://api.esios.ree.es"

# ---------------------------------------------------------------
# 2) MAPEO: columna solicitada -> texto de búsqueda en ESIOS
#    (usamos "Generación T.Real <tecnología>" que es la serie medida
#     en tiempo real, ámbito peninsular, publicada por REE)
# ---------------------------------------------------------------
MAPEO_BUSQUEDA = {
    "biogas": "Generación T.Real Biogás",
    "biomasa": "Generación T.Real Biomasa",
    "ciclo_combinado": "Generación T.Real Ciclo combinado",
    "derivados_del_petroleo_o_carbon": "Generación T.Real Derivados del petróleo o carbón",
    "energia_residual": "Generación T.Real Energía residual",
    "eolica_terrestre": "Generación T.Real Eólica terrestre",
    "fuel": "Generación T.Real Fuel",
    "gas_natural_cogeneracion": "Generación T.Real Cogeneración",
    "hidraulica_no_ugh": "Generación T.Real Hidráulica no UGH",
    "hidraulica_ugh": "Generación T.Real Hidráulica UGH",
    "hulla_antracita": "Generación T.Real Hulla antracita",
    "hulla_sub_bituminosa": "Generación T.Real Hulla sub-bituminosa",
    "nuclear": "Generación T.Real Nuclear",
    "oceano_y_geotermica": "Generación T.Real Geotérmica",
    "residuos_domesticos_y_similares": "Generación T.Real Residuos domésticos",
    "residuos_varios": "Generación T.Real Residuos varios",
    "solar_fotovoltaica": "Generación T.Real Solar fotovoltaica",
    "solar_termica": "Generación T.Real Solar térmica",
    "subproductos_mineria": "Generación T.Real Subproductos minería",
    "termica_no_renovable": "Generación T.Real Térmica renovable",  # revisar/ajustar
    "total_tipo_produccion": "Generación T.Real total",
    "turbinacion_bombeo": "Generación T.Real Turbinación bombeo",
}


def buscar_id_indicador(texto_busqueda):
    """Busca en el catálogo de ESIOS el indicador cuyo nombre coincide mejor."""
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": texto_busqueda},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])
    if not data:
        return None, None
    # Nos quedamos con el primer resultado (más relevante) y devolvemos también
    # el nombre real para que puedas verificar que es correcto
    return data[0]["id"], data[0]["name"]


def descargar_indicador(indicator_id):
    resp = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={
            "start_date": START_DATE,
            "end_date": END_DATE,
            "geo_ids[]": 8741,  # sistema peninsular
        },
        timeout=30,
    )
    resp.raise_for_status()
    valores = resp.json()["indicator"]["values"]
    df = pd.DataFrame(valores)[["datetime", "value"]]
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)
    return df.rename(columns={"datetime": "fecha", "value": "valor"})


if __name__ == "__main__":
    if TOKEN == "TU_TOKEN_AQUI":
        raise SystemExit(
            "Falta configurar tu token de ESIOS. Solicítalo en consultasios@ree.es "
            "y pégalo en la variable TOKEN."
        )

    dataset = None
    ids_encontrados = {}

    for columna, texto in MAPEO_BUSQUEDA.items():
        print(f"Buscando indicador para: {columna} ('{texto}')...")
        ind_id, nombre_real = buscar_id_indicador(texto)

        if ind_id is None:
            print(f"  -> No se encontró ningún indicador para '{texto}'. Se omite.")
            continue

        print(f"  -> Encontrado: id={ind_id} | nombre='{nombre_real}'")
        ids_encontrados[columna] = (ind_id, nombre_real)

        try:
            df_ind = descargar_indicador(ind_id)
        except Exception as e:
            print(f"  -> Error descargando id {ind_id}: {e}")
            continue

        df_ind = df_ind.rename(columns={"valor": columna}).set_index("fecha")

        if dataset is None:
            dataset = df_ind
        else:
            dataset = dataset.join(df_ind, how="outer")

        time.sleep(0.3)  # evitar saturar la API

    if dataset is not None:
        dataset = dataset.sort_index().reset_index()
        salida = "generacion_detallada_peninsula_2020Q4.csv"
        dataset.to_csv(salida, index=False, encoding="utf-8-sig")
        print(f"\nDataset guardado en: {salida}")
        print(f"Columnas obtenidas: {list(dataset.columns)}")

        print("\nMapeo columna -> indicador ESIOS usado (revisa que sea correcto):")
        for col, (idi, nom) in ids_encontrados.items():
            print(f"  {col}: id={idi} -> '{nom}'")
    else:
        print("No se pudo construir el dataset. Revisa el token y las búsquedas.")

Buscando indicador para: biogas ('Generación T.Real Biogás')...
  -> No se encontró ningún indicador para 'Generación T.Real Biogás'. Se omite.
Buscando indicador para: biomasa ('Generación T.Real Biomasa')...
  -> No se encontró ningún indicador para 'Generación T.Real Biomasa'. Se omite.
Buscando indicador para: ciclo_combinado ('Generación T.Real Ciclo combinado')...
  -> Encontrado: id=2041 | nombre='Generación T.Real ciclo combinado nacional'
  -> Error descargando id 2041: "None of [Index(['datetime', 'value'], dtype='str')] are in the [columns]"
Buscando indicador para: derivados_del_petroleo_o_carbon ('Generación T.Real Derivados del petróleo o carbón')...
  -> No se encontró ningún indicador para 'Generación T.Real Derivados del petróleo o carbón'. Se omite.
Buscando indicador para: energia_residual ('Generación T.Real Energía residual')...
  -> No se encontró ningún indicador para 'Generación T.Real Energía residual'. Se omite.
Buscando indicador para: eolica_terrestre ('Gene

## Version 2

Qué cambia respecto a la versión anterior:

Busca ahora Generación programada PBF <tipo> en vez de T.Real ... nacional — esta es la serie correcta: desagregada por tipo de producción y a nivel peninsular, con histórico completo desde hace años.

Si un indicador devuelve 0 valores o una estructura rara, ahora te lo dice de forma clara (ValueError) en vez de romper el script.

La columna total_tipo_produccion ya no se busca como indicador (no existe como tal) — se calcula automáticamente sumando el resto de columnas descargadas.

Algunos textos de búsqueda (Otras renovables para oceano_y_geotermica, Térmica renovable, Cogeneración con gas natural) son mi mejor aproximación al nombre oficial — revisa el "nombre real" que imprime el script para cada uno y avísame si alguno no encaja, así ajustamos el texto de búsqueda.

In [3]:
"""
Descarga de generación eléctrica detallada por tipo de tecnología
Sistema peninsular, 01/10/2020 - 31/12/2020

Fuente: API de ESIOS (Red Eléctrica de España)
Requiere un token personal gratuito: solicítalo a consultasios@ree.es
Documentación: https://api.esios.ree.es/

El script busca automáticamente el ID de cada indicador por nombre,
descarga sus valores horarios y construye un dataset ancho con las
columnas solicitadas.
"""

import requests
import pandas as pd
import time

# ---------------------------------------------------------------
# 1) CONFIGURACIÓN
# ---------------------------------------------------------------
TOKEN = "294da85af17aff04fda25ec930700cf59bde0ac3d8322d075617fc038c1ff224"   # <-- pega aquí tu token personal de ESIOS

START_DATE = "2020-10-01T00:00:00"
END_DATE   = "2020-12-31T23:59:59"

HEADERS = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json",
    "x-api-key": TOKEN,
}

BASE_URL = "https://api.esios.ree.es"

# ---------------------------------------------------------------
# 2) MAPEO: columna solicitada -> texto de búsqueda en ESIOS
#    (usamos "Generación T.Real <tecnología>" que es la serie medida
#     en tiempo real, ámbito peninsular, publicada por REE)
# ---------------------------------------------------------------
MAPEO_BUSQUEDA = {
    "biogas": "Generación programada PBF Biogás",
    "biomasa": "Generación programada PBF Biomasa",
    "ciclo_combinado": "Generación programada PBF Ciclo combinado",
    "derivados_del_petroleo_o_carbon": "Generación programada PBF Derivados de petróleo o carbón",
    "energia_residual": "Generación programada PBF Energía residual",
    "eolica_terrestre": "Generación programada PBF Eólica terrestre",
    "fuel": "Generación programada PBF Fuel-gas",
    "gas_natural_cogeneracion": "Generación programada PBF Cogeneración con gas natural",
    "hidraulica_no_ugh": "Generación programada PBF Hidráulica no UGH",
    "hidraulica_ugh": "Generación programada PBF Hidráulica UGH",
    "hulla_antracita": "Generación programada PBF Hulla antracita",
    "hulla_sub_bituminosa": "Generación programada PBF Hulla sub-bituminosa",
    "nuclear": "Generación programada PBF Nuclear",
    "oceano_y_geotermica": "Generación programada PBF Otras renovables",
    "residuos_domesticos_y_similares": "Generación programada PBF Residuos domésticos",
    "residuos_varios": "Generación programada PBF Residuos varios",
    "solar_fotovoltaica": "Generación programada PBF Solar fotovoltaica",
    "solar_termica": "Generación programada PBF Solar térmica",
    "subproductos_mineria": "Generación programada PBF Subproductos minería",
    "termica_no_renovable": "Generación programada PBF Térmica renovable",
    "total_tipo_produccion": None,  # se calcula como suma de todo lo demás
    "turbinacion_bombeo": "Generación programada PBF Turbinación bombeo",
}


def buscar_id_indicador(texto_busqueda):
    """Busca en el catálogo de ESIOS el indicador cuyo nombre coincide mejor."""
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": texto_busqueda},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])
    if not data:
        return None, None
    # Nos quedamos con el primer resultado (más relevante) y devolvemos también
    # el nombre real para que puedas verificar que es correcto
    return data[0]["id"], data[0]["name"]


def descargar_indicador(indicator_id):
    resp = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={
            "start_date": START_DATE,
            "end_date": END_DATE,
        },
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()
    valores = payload.get("indicator", {}).get("values", [])

    if not valores:
        raise ValueError("La API devolvió 0 valores para este indicador y rango de fechas.")

    df = pd.DataFrame(valores)

    # Si el indicador tiene varios geo_ids (p.ej. varias CCAA), nos quedamos
    # con el ámbito peninsular (geo_id 8741) si existe esa columna
    if "geo_id" in df.columns and (df["geo_id"] == 8741).any():
        df = df[df["geo_id"] == 8741]

    if "datetime" not in df.columns or "value" not in df.columns:
        raise ValueError(f"Estructura inesperada. Columnas recibidas: {list(df.columns)}")

    df = df[["datetime", "value"]].copy()
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)
    return df.rename(columns={"datetime": "fecha", "value": "valor"})


if __name__ == "__main__":
    if TOKEN == "TU_TOKEN_AQUI":
        raise SystemExit(
            "Falta configurar tu token de ESIOS. Solicítalo en consultasios@ree.es "
            "y pégalo en la variable TOKEN."
        )

    dataset = None
    ids_encontrados = {}

    for columna, texto in MAPEO_BUSQUEDA.items():
        if texto is None:
            print(f"Columna '{columna}': se calculará como suma del resto al final. Se omite búsqueda.")
            continue

        print(f"Buscando indicador para: {columna} ('{texto}')...")
        ind_id, nombre_real = buscar_id_indicador(texto)

        if ind_id is None:
            print(f"  -> No se encontró ningún indicador para '{texto}'. Se omite.")
            continue

        print(f"  -> Encontrado: id={ind_id} | nombre='{nombre_real}'")
        ids_encontrados[columna] = (ind_id, nombre_real)

        try:
            df_ind = descargar_indicador(ind_id)
        except Exception as e:
            print(f"  -> Error descargando id {ind_id}: {e}")
            continue

        df_ind = df_ind.rename(columns={"valor": columna}).set_index("fecha")

        if dataset is None:
            dataset = df_ind
        else:
            dataset = dataset.join(df_ind, how="outer")

        time.sleep(0.3)  # evitar saturar la API

    if dataset is not None:
        dataset = dataset.sort_index().reset_index()

        columnas_tecnologia = [c for c in dataset.columns if c != "fecha"]
        if "total_tipo_produccion" in MAPEO_BUSQUEDA:
            dataset["total_tipo_produccion"] = dataset[columnas_tecnologia].sum(axis=1, skipna=True)
        salida = "generacion_detallada_peninsula_2020Q4.csv"
        dataset.to_csv(salida, index=False, encoding="utf-8-sig")
        print(f"\nDataset guardado en: {salida}")
        print(f"Columnas obtenidas: {list(dataset.columns)}")

        print("\nMapeo columna -> indicador ESIOS usado (revisa que sea correcto):")
        for col, (idi, nom) in ids_encontrados.items():
            print(f"  {col}: id={idi} -> '{nom}'")
    else:
        print("No se pudo construir el dataset. Revisa el token y las búsquedas.")

Buscando indicador para: biogas ('Generación programada PBF Biogás')...
  -> Encontrado: id=22 | nombre='Generación programada PBF Biogás'
Buscando indicador para: biomasa ('Generación programada PBF Biomasa')...
  -> Encontrado: id=21 | nombre='Generación programada PBF Biomasa'
Buscando indicador para: ciclo_combinado ('Generación programada PBF Ciclo combinado')...
  -> Encontrado: id=9 | nombre='Generación programada PBF Ciclo combinado'
Buscando indicador para: derivados_del_petroleo_o_carbon ('Generación programada PBF Derivados de petróleo o carbón')...
  -> No se encontró ningún indicador para 'Generación programada PBF Derivados de petróleo o carbón'. Se omite.
Buscando indicador para: energia_residual ('Generación programada PBF Energía residual')...
  -> Encontrado: id=20 | nombre='Generación programada PBF Energía residual'
Buscando indicador para: eolica_terrestre ('Generación programada PBF Eólica terrestre')...
  -> Encontrado: id=12 | nombre='Generación programada PBF E

In [2]:
"""
Descarga de generación eléctrica detallada por tipo de tecnología
Sistema peninsular, 01/10/2020 - 31/12/2020

Fuente: API de ESIOS (Red Eléctrica de España)
Requiere un token personal gratuito: solicítalo a consultasios@ree.es
Documentación: https://api.esios.ree.es/

El script busca automáticamente el ID de cada indicador por nombre,
descarga sus valores horarios y construye un dataset ancho con las
columnas solicitadas.
"""

import requests
import pandas as pd
import time

# ---------------------------------------------------------------
# 1) CONFIGURACIÓN
# ---------------------------------------------------------------
TOKEN = "294da85af17aff04fda25ec930700cf59bde0ac3d8322d075617fc038c1ff224"   # <-- pega aquí tu token personal de ESIOS

START_DATE = "2020-10-01T00:00:00"
END_DATE   = "2020-12-31T23:59:59"

HEADERS = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json",
    "x-api-key": TOKEN,
}

BASE_URL = "https://api.esios.ree.es"

# ---------------------------------------------------------------
# 2) MAPEO: columna solicitada -> texto de búsqueda en ESIOS
#    (usamos "Generación T.Real <tecnología>" que es la serie medida
#     en tiempo real, ámbito peninsular, publicada por REE)
# ---------------------------------------------------------------
MAPEO_BUSQUEDA = {
    "biogas": "Generación programada PBF Biogás",
    "biomasa": "Generación programada PBF Biomasa",
    "ciclo_combinado": "Generación programada PBF Ciclo combinado",
    "derivados_del_petroleo_o_carbon": "Generación programada PBF Derivados de petróleo o carbón",
    "energia_residual": "Generación programada PBF Energía residual",
    "eolica_terrestre": "Generación programada PBF Eólica terrestre",
    "fuel": "Generación programada PBF Fuel-gas",
    "gas_natural_cogeneracion": "Generación programada PBF Cogeneración con gas natural",
    "hidraulica_no_ugh": "Generación programada PBF Hidráulica no UGH",
    "hidraulica_ugh": "Generación programada PBF Hidráulica UGH",
    "hulla_antracita": "Generación programada PBF Hulla antracita",
    "hulla_sub_bituminosa": "Generación programada PBF Hulla sub-bituminosa",
    "nuclear": "Generación programada PBF Nuclear",
    "oceano_y_geotermica": "Generación programada PBF Otras renovables",
    "residuos_domesticos_y_similares": "Generación programada PBF Residuos domésticos",
    "residuos_varios": "Generación programada PBF Residuos varios",
    "solar_fotovoltaica": "Generación programada PBF Solar fotovoltaica",
    "solar_termica": "Generación programada PBF Solar térmica",
    "subproductos_mineria": "Generación programada PBF Subproductos minería",
    "termica_no_renovable": "Generación programada PBF Térmica renovable",
    "total_tipo_produccion": None,  # se calcula como suma de todo lo demás
    "turbinacion_bombeo": "Generación programada PBF Turbinación bombeo",
}


def listar_catalogo_pbf():
    """Lista TODOS los indicadores cuyo nombre contiene 'Generación programada PBF',
    para poder ver el catálogo completo y elegir a mano los que falten."""
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": "Generación programada PBF"},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])
    print(f"\n=== Catálogo completo 'Generación programada PBF' ({len(data)} indicadores) ===")
    for ind in sorted(data, key=lambda x: x["id"] if isinstance(x["id"], int) else 999999):
        print(f"  id={ind['id']:<8} {ind['name']}")
    print("=== fin catálogo ===\n")
    return data


def buscar_id_indicador(texto_busqueda):
    """Busca en el catálogo de ESIOS el indicador cuyo nombre coincide mejor.
    Prioriza coincidencia EXACTA (ignorando mayúsculas/acentos simples) antes
    de quedarse con el primer resultado, para evitar falsos positivos como
    'Hidráulica UGH' emparejando con 'Hidráulica no UGH'."""
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": texto_busqueda},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])
    if not data:
        return None, None

    objetivo = texto_busqueda.strip().lower()
    for ind in data:
        if ind["name"].strip().lower() == objetivo:
            return ind["id"], ind["name"]

    # si no hay coincidencia exacta, preferimos la más corta que contenga
    # el texto buscado (suele ser la más específica, no una variante larga)
    candidatas = [ind for ind in data if objetivo in ind["name"].strip().lower()]
    if candidatas:
        mejor = min(candidatas, key=lambda x: len(x["name"]))
        return mejor["id"], mejor["name"]

    return data[0]["id"], data[0]["name"]


def descargar_indicador(indicator_id):
    resp = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={
            "start_date": START_DATE,
            "end_date": END_DATE,
        },
        timeout=30,
    )
    resp.raise_for_status()
    payload = resp.json()
    valores = payload.get("indicator", {}).get("values", [])

    if not valores:
        raise ValueError("La API devolvió 0 valores para este indicador y rango de fechas.")

    df = pd.DataFrame(valores)

    # Si el indicador tiene varios geo_ids (p.ej. varias CCAA), nos quedamos
    # con el ámbito peninsular (geo_id 8741) si existe esa columna
    if "geo_id" in df.columns and (df["geo_id"] == 8741).any():
        df = df[df["geo_id"] == 8741]

    if "datetime" not in df.columns or "value" not in df.columns:
        raise ValueError(f"Estructura inesperada. Columnas recibidas: {list(df.columns)}")

    df = df[["datetime", "value"]].copy()
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)
    return df.rename(columns={"datetime": "fecha", "value": "valor"})


if __name__ == "__main__":
    if TOKEN == "TU_TOKEN_AQUI":
        raise SystemExit(
            "Falta configurar tu token de ESIOS. Solicítalo en consultasios@ree.es "
            "y pégalo en la variable TOKEN."
        )

    listar_catalogo_pbf()

    dataset = None
    ids_encontrados = {}

    for columna, texto in MAPEO_BUSQUEDA.items():
        if texto is None:
            print(f"Columna '{columna}': se calculará como suma del resto al final. Se omite búsqueda.")
            continue

        print(f"Buscando indicador para: {columna} ('{texto}')...")
        ind_id, nombre_real = buscar_id_indicador(texto)

        if ind_id is None:
            print(f"  -> No se encontró ningún indicador para '{texto}'. Se omite.")
            continue

        print(f"  -> Encontrado: id={ind_id} | nombre='{nombre_real}'")
        ids_encontrados[columna] = (ind_id, nombre_real)

        try:
            df_ind = descargar_indicador(ind_id)
        except Exception as e:
            print(f"  -> Error descargando id {ind_id}: {e}")
            continue

        df_ind = df_ind.rename(columns={"valor": columna}).set_index("fecha")

        if dataset is None:
            dataset = df_ind
        else:
            dataset = dataset.join(df_ind, how="outer")

        time.sleep(0.3)  # evitar saturar la API

    if dataset is not None:
        dataset = dataset.sort_index().reset_index()

        columnas_tecnologia = [c for c in dataset.columns if c != "fecha"]
        if "total_tipo_produccion" in MAPEO_BUSQUEDA:
            dataset["total_tipo_produccion"] = dataset[columnas_tecnologia].sum(axis=1, skipna=True)
        salida = "generacion_detallada_peninsula_2020Q4.csv"
        dataset.to_csv(salida, index=False, encoding="utf-8-sig")
        print(f"\nDataset guardado en: {salida}")
        print(f"Columnas obtenidas: {list(dataset.columns)}")

        print("\nMapeo columna -> indicador ESIOS usado (revisa que sea correcto):")
        for col, (idi, nom) in ids_encontrados.items():
            print(f"  {col}: id={idi} -> '{nom}'")
    else:
        print("No se pudo construir el dataset. Revisa el token y las búsquedas.")


=== Catálogo completo 'Generación programada PBF' (50 indicadores) ===
  id=1        Generación programada PBF Hidráulica UGH
  id=2        Generación programada PBF Hidráulica no UGH
  id=3        Generación programada PBF Turbinación bombeo
  id=4        Generación programada PBF Nuclear
  id=5        Generación programada PBF Hulla antracita Anexo II RD 134/2010
  id=6        Generación programada PBF Hulla sub-bituminosa Anexo II RD 134/2010
  id=7        Generación programada PBF Hulla antracita
  id=8        Generación programada PBF Hulla sub-bituminosa
  id=9        Generación programada PBF Ciclo combinado
  id=10       Generación programada PBF Fuel
  id=11       Generación programada PBF Gas Natural - Turbina de vapor
  id=12       Generación programada PBF Eólica terrestre
  id=13       Generación programada PBF Eólica marina
  id=14       Generación programada PBF Solar fotovoltaica
  id=15       Generación programada PBF Solar térmica
  id=16       Generación programada 

1. ❌ derivados_del_petroleo_o_carbon

Buscaste:

Generación programada PBF Derivados de petróleo o carbón

Pero en el catálogo aparece como:

Generación programada PBF Derivados del petróleo ó carbón

2. ❌ gas_natural_cogeneracion

Buscaste:

Cogeneración con gas natural

Pero el real es:

Generación programada PBF Gas Natural Cogeneración

3. ❌ termica_no_renovable

Buscaste:

Térmica renovable

👉 Esto está mal conceptualmente.

No existe “térmica no renovable” como tal en PBF.

✔️ Reconstruir termica_no_renovable a partir de los otros datos

4. ⚠️ fuel

Estás usando:

id=10077 → Fuel-Gas (agregado moderno)

👉 Problema: no tiene datos en 2020Q4

✔ Correcto:

id=10 → Generación programada PBF Fuel


5. ✔️ Mejora robusta del buscador

Tu función falla con acentos y variantes. Mejorar


In [6]:
import requests
import pandas as pd
import time
import unicodedata

# ---------------------------------------------------------------
# 1) CONFIGURACIÓN
# ---------------------------------------------------------------
TOKEN = "294da85af17aff04fda25ec930700cf59bde0ac3d8322d075617fc038c1ff224"

START_DATE = "2020-10-01T00:00:00"
END_DATE   = "2020-12-31T23:59:59"

HEADERS = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json",
    "x-api-key": TOKEN,
}

BASE_URL = "https://api.esios.ree.es"

# ---------------------------------------------------------------
# 2) MAPEO CORREGIDO
# ---------------------------------------------------------------
MAPEO_BUSQUEDA = {
    "biogas": "Generación programada PBF Biogás",
    "biomasa": "Generación programada PBF Biomasa",
    "ciclo_combinado": "Generación programada PBF Ciclo combinado",

    # corregido
    "derivados_del_petroleo_o_carbon": "Generación programada PBF Derivados del petróleo ó carbón",

    "energia_residual": "Generación programada PBF Energía residual",
    "eolica_terrestre": "Generación programada PBF Eólica terrestre",

    # usar fuel correcto
    "fuel": "Generación programada PBF Fuel",

    # corregido
    "gas_natural_cogeneracion": "Generación programada PBF Gas Natural Cogeneración",

    "hidraulica_no_ugh": "Generación programada PBF Hidráulica no UGH",
    "hidraulica_ugh": "Generación programada PBF Hidráulica UGH",

    "hulla_antracita": "Generación programada PBF Hulla antracita",
    "hulla_sub_bituminosa": "Generación programada PBF Hulla sub-bituminosa",

    "nuclear": "Generación programada PBF Nuclear",

    # mejor específico
    "oceano_y_geotermica": "Generación programada PBF Océano y geotérmica",

    "residuos_domesticos_y_similares": "Generación programada PBF Residuos domésticos y similares",
    "residuos_varios": "Generación programada PBF Residuos varios",

    "solar_fotovoltaica": "Generación programada PBF Solar fotovoltaica",
    "solar_termica": "Generación programada PBF Solar térmica",

    "subproductos_mineria": "Generación programada PBF Subproductos minería",

    # se calcula luego (opcional)
    "termica_no_renovable": None,

    "total_tipo_produccion": None,
    "turbinacion_bombeo": "Generación programada PBF Turbinación bombeo",
}

# ---------------------------------------------------------------
# NORMALIZACIÓN (clave para que funcione bien)
# ---------------------------------------------------------------
def normalizar(texto):
    texto = texto.lower()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto


def listar_catalogo_pbf():
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": "Generación programada PBF"},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])

    print(f"\n=== Catálogo completo ({len(data)} indicadores) ===")
    for ind in sorted(data, key=lambda x: x["id"] if isinstance(x["id"], int) else 999999):
        print(f"  id={ind['id']:<8} {ind['name']}")
    print("=== fin catálogo ===\n")

    return data


def buscar_id_indicador(texto_busqueda):
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": texto_busqueda},
        timeout=30,
    )
    resp.raise_for_status()

    data = resp.json().get("indicators", [])
    if not data:
        return None, None

    objetivo = normalizar(texto_busqueda)

    # 1) coincidencia exacta normalizada
    for ind in data:
        if normalizar(ind["name"]) == objetivo:
            return ind["id"], ind["name"]

    # 2) contiene texto
    candidatas = [ind for ind in data if objetivo in normalizar(ind["name"])]
    if candidatas:
        mejor = min(candidatas, key=lambda x: len(x["name"]))
        return mejor["id"], mejor["name"]

    # 3) fallback
    return data[0]["id"], data[0]["name"]


def descargar_indicador(indicator_id):
    resp = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={
            "start_date": START_DATE,
            "end_date": END_DATE,
        },
        timeout=30,
    )
    resp.raise_for_status()

    payload = resp.json()
    valores = payload.get("indicator", {}).get("values", [])

    if not valores:
        raise ValueError("0 valores en este rango de fechas")

    df = pd.DataFrame(valores)

    if "geo_id" in df.columns and (df["geo_id"] == 8741).any():
        df = df[df["geo_id"] == 8741]

    df = df[["datetime", "value"]].copy()
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

    return df.rename(columns={"datetime": "fecha", "value": "valor"})


# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------
if __name__ == "__main__":
    if TOKEN == "TU_TOKEN_AQUI":
        raise SystemExit("Configura tu TOKEN de ESIOS")

    listar_catalogo_pbf()

    dataset = None
    ids_encontrados = {}

    for columna, texto in MAPEO_BUSQUEDA.items():

        if texto is None:
            print(f"{columna}: se calculará después")
            continue

        print(f"\nBuscando: {columna}")
        ind_id, nombre_real = buscar_id_indicador(texto)

        if ind_id is None:
            print("  -> no encontrado")
            continue

        print(f"  -> id={ind_id} | {nombre_real}")
        ids_encontrados[columna] = (ind_id, nombre_real)

        try:
            df_ind = descargar_indicador(ind_id)
        except Exception as e:
            print(f"  -> error: {e}")
            continue

        df_ind = df_ind.rename(columns={"valor": columna}).set_index("fecha")

        if dataset is None:
            dataset = df_ind
        else:
            dataset = dataset.join(df_ind, how="outer")

        time.sleep(0.3)

    if dataset is not None:
        dataset = dataset.sort_index().reset_index()

        # total
        columnas_tecnologia = [c for c in dataset.columns if c != "fecha"]
        dataset["total_tipo_produccion"] = dataset[columnas_tecnologia].sum(axis=1, skipna=True)

        # opcional: térmica no renovable
        cols_termicas = [
            "ciclo_combinado",
            "fuel",
            "gas_natural_cogeneracion",
            "hulla_antracita",
            "hulla_sub_bituminosa",
            "derivados_del_petroleo_o_carbon",
        ]

        existentes = [c for c in cols_termicas if c in dataset.columns]
        if existentes:
            dataset["termica_no_renovable"] = dataset[existentes].sum(axis=1, skipna=True)

        salida = "generacion_detallada_peninsula_2020Q4.csv"
        dataset.to_csv(salida, index=False, encoding="utf-8-sig")

        print(f"\nGuardado en: {salida}")
        print("Columnas:", list(dataset.columns))

    else:
        print("No se generó dataset")


=== Catálogo completo (50 indicadores) ===
  id=1        Generación programada PBF Hidráulica UGH
  id=2        Generación programada PBF Hidráulica no UGH
  id=3        Generación programada PBF Turbinación bombeo
  id=4        Generación programada PBF Nuclear
  id=5        Generación programada PBF Hulla antracita Anexo II RD 134/2010
  id=6        Generación programada PBF Hulla sub-bituminosa Anexo II RD 134/2010
  id=7        Generación programada PBF Hulla antracita
  id=8        Generación programada PBF Hulla sub-bituminosa
  id=9        Generación programada PBF Ciclo combinado
  id=10       Generación programada PBF Fuel
  id=11       Generación programada PBF Gas Natural - Turbina de vapor
  id=12       Generación programada PBF Eólica terrestre
  id=13       Generación programada PBF Eólica marina
  id=14       Generación programada PBF Solar fotovoltaica
  id=15       Generación programada PBF Solar térmica
  id=16       Generación programada PBF Océano y geotérmica
  id

## Solucoinar Probelma de Fuel

No se obtiene nigunn dato, porque la produccion de energia mediante este tipo es 0.00 
Solucion: rellenar el dato con 0

Considerar eliminar este dato si en todo el dataset durante los 5 años siempre es 0.00 (no aporta nada)

In [ ]:
import requests
import pandas as pd
import time
import unicodedata

# ---------------------------------------------------------------
# 1) CONFIGURACIÓN
# ---------------------------------------------------------------
TOKEN = "294da85af17aff04fda25ec930700cf59bde0ac3d8322d075617fc038c1ff224"

START_DATE = "2020-10-01T00:00:00"
END_DATE   = "2020-12-31T23:59:59"

HEADERS = {
    "Accept": "application/json; application/vnd.api+json",
    "Content-Type": "application/json",
    "x-api-key": TOKEN,
}

BASE_URL = "https://api.esios.ree.es"

# ---------------------------------------------------------------
# 2) MAPEO CORREGIDO
# ---------------------------------------------------------------
MAPEO_BUSQUEDA = {
    "biogas": "Generación programada PBF Biogás",
    "biomasa": "Generación programada PBF Biomasa",
    "ciclo_combinado": "Generación programada PBF Ciclo combinado",

    # corregido
    "derivados_del_petroleo_o_carbon": "Generación programada PBF Derivados del petróleo ó carbón",

    "energia_residual": "Generación programada PBF Energía residual",
    "eolica_terrestre": "Generación programada PBF Eólica terrestre",

    # usar fuel correcto
    "fuel": "Generación programada PBF Fuel",

    # corregido
    "gas_natural_cogeneracion": "Generación programada PBF Gas Natural Cogeneración",

    "hidraulica_no_ugh": "Generación programada PBF Hidráulica no UGH",
    "hidraulica_ugh": "Generación programada PBF Hidráulica UGH",

    "hulla_antracita": "Generación programada PBF Hulla antracita",
    "hulla_sub_bituminosa": "Generación programada PBF Hulla sub-bituminosa",

    "nuclear": "Generación programada PBF Nuclear",

    # mejor específico
    "oceano_y_geotermica": "Generación programada PBF Océano y geotérmica",

    "residuos_domesticos_y_similares": "Generación programada PBF Residuos domésticos y similares",
    "residuos_varios": "Generación programada PBF Residuos varios",

    "solar_fotovoltaica": "Generación programada PBF Solar fotovoltaica",
    "solar_termica": "Generación programada PBF Solar térmica",

    "subproductos_mineria": "Generación programada PBF Subproductos minería",

    # se calcula luego (opcional)
    "termica_no_renovable": None,

    "total_tipo_produccion": None,
    "turbinacion_bombeo": "Generación programada PBF Turbinación bombeo",
}

# ---------------------------------------------------------------
# NORMALIZACIÓN (clave para que funcione bien)
# ---------------------------------------------------------------
def normalizar(texto):
    texto = texto.lower()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto


def listar_catalogo_pbf():
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": "Generación programada PBF"},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json().get("indicators", [])

    print(f"\n=== Catálogo completo ({len(data)} indicadores) ===")
    for ind in sorted(data, key=lambda x: x["id"] if isinstance(x["id"], int) else 999999):
        print(f"  id={ind['id']:<8} {ind['name']}")
    print("=== fin catálogo ===\n")

    return data


def buscar_id_indicador(texto_busqueda):
    resp = requests.get(
        f"{BASE_URL}/indicators",
        headers=HEADERS,
        params={"text": texto_busqueda},
        timeout=30,
    )
    resp.raise_for_status()

    data = resp.json().get("indicators", [])
    if not data:
        return None, None

    objetivo = normalizar(texto_busqueda)

    # 1) coincidencia exacta normalizada
    for ind in data:
        if normalizar(ind["name"]) == objetivo:
            return ind["id"], ind["name"]

    # 2) contiene texto
    candidatas = [ind for ind in data if objetivo in normalizar(ind["name"])]
    if candidatas:
        mejor = min(candidatas, key=lambda x: len(x["name"]))
        return mejor["id"], mejor["name"]

    # 3) fallback
    return data[0]["id"], data[0]["name"]


def descargar_indicador(indicator_id):
    resp = requests.get(
        f"{BASE_URL}/indicators/{indicator_id}",
        headers=HEADERS,
        params={
            "start_date": START_DATE,
            "end_date": END_DATE,
        },
        timeout=30,
    )
    resp.raise_for_status()

    payload = resp.json()
    valores = payload.get("indicator", {}).get("values", [])

    if not valores:
        raise ValueError("0 valores en este rango de fechas")

    df = pd.DataFrame(valores)

    if "geo_id" in df.columns and (df["geo_id"] == 8741).any():
        df = df[df["geo_id"] == 8741]

    df = df[["datetime", "value"]].copy()
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_localize(None)

    return df.rename(columns={"datetime": "fecha", "value": "valor"})


# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------
if __name__ == "__main__":
    if TOKEN == "TU_TOKEN_AQUI":
        raise SystemExit("Configura tu TOKEN de ESIOS")

    listar_catalogo_pbf()

    dataset = None
    ids_encontrados = {}

    for columna, texto in MAPEO_BUSQUEDA.items():

        if texto is None:
            print(f"{columna}: se calculará después")
            continue

        print(f"\nBuscando: {columna}")
        ind_id, nombre_real = buscar_id_indicador(texto)

        if ind_id is None:
            print("  -> no encontrado")
            continue

        print(f"  -> id={ind_id} | {nombre_real}")
        ids_encontrados[columna] = (ind_id, nombre_real)

        try:
            df_ind = descargar_indicador(ind_id)
        except Exception as e:
            print(f"  -> error: {e}")
            continue

        df_ind = df_ind.rename(columns={"valor": columna}).set_index("fecha")

        if dataset is None:
            dataset = df_ind
        else:
            dataset = dataset.join(df_ind, how="outer")

        time.sleep(0.3)

    if dataset is not None:
        dataset = dataset.sort_index().reset_index()

        # total
        columnas_tecnologia = [c for c in dataset.columns if c != "fecha"]
        dataset["total_tipo_produccion"] = dataset[columnas_tecnologia].sum(axis=1, skipna=True)

        # opcional: térmica no renovable
        cols_termicas = [
            "ciclo_combinado",
            "fuel",
            "gas_natural_cogeneracion",
            "hulla_antracita",
            "hulla_sub_bituminosa",
            "derivados_del_petroleo_o_carbon",
        ]

        existentes = [c for c in cols_termicas if c in dataset.columns]
        if existentes:
            dataset["termica_no_renovable"] = dataset[existentes].sum(axis=1, skipna=True)

        if "fuel" not in dataset.columns:
            dataset["fuel"] = 0.0
            dataset["fuel"] = dataset["fuel"].fillna(0.0)

        salida = "generacion_detallada_peninsula_2020Q4.csv"
        dataset.to_csv(salida, index=False, encoding="utf-8-sig")

        print(f"\nGuardado en: {salida}")
        print("Columnas:", list(dataset.columns))

    else:
        print("No se generó dataset")


=== Catálogo completo (50 indicadores) ===
  id=1        Generación programada PBF Hidráulica UGH
  id=2        Generación programada PBF Hidráulica no UGH
  id=3        Generación programada PBF Turbinación bombeo
  id=4        Generación programada PBF Nuclear
  id=5        Generación programada PBF Hulla antracita Anexo II RD 134/2010
  id=6        Generación programada PBF Hulla sub-bituminosa Anexo II RD 134/2010
  id=7        Generación programada PBF Hulla antracita
  id=8        Generación programada PBF Hulla sub-bituminosa
  id=9        Generación programada PBF Ciclo combinado
  id=10       Generación programada PBF Fuel
  id=11       Generación programada PBF Gas Natural - Turbina de vapor
  id=12       Generación programada PBF Eólica terrestre
  id=13       Generación programada PBF Eólica marina
  id=14       Generación programada PBF Solar fotovoltaica
  id=15       Generación programada PBF Solar térmica
  id=16       Generación programada PBF Océano y geotérmica
  id